In [39]:
import pickle 
import re

from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import SparsePauliOp
from qiskit.circuit.library import CXGate, PauliEvolutionGate
from qiskit.transpiler import PassManager
from qiskit.transpiler.passes import InverseCancellation, CommutativeCancellation
from qopt_best_practices.transpilation.swap_cancellation_pass import SwapToFinalMapping

from qiskit_qaoa.utils.transpiler_passes import ExtendedSwapStrategy, FindCommutingPauliEvolutionsMulti
from qiskit_qaoa.utils.commuting_gate_router import CommutingGateRouter
from qiskit_qaoa.utils.gfa_utils import gfa_file_to_graph
from qiskit_qaoa.hubo.graph_to_hubo_hamiltonian import graph_to_hubo_hamiltonian

In [11]:
def print_circuit_info(qc: QuantumCircuit, circuit_name):
    print(f'{circuit_name} has {qc.num_qubits} qubits')
    print(f'{circuit_name} has {qc.num_nonlocal_gates()} non-local gates and {qc.depth(lambda instr: len(instr.qubits) > 1)} non-local depth')
    print(f'{circuit_name} contains {list(qc.count_ops().keys())} gates.')

In [12]:
hubo_hams = {}
hubo_circs = {}
hubo_reordered_hams = {}
results = {}
for name, filename, copy_numbers in zip(
    [
        'H_6', 'H_9', 'H_12', 
        'H_20(1)', 'H_20(2)', 'H_30'
    ],
    [
        'test_N2_W2', 'trivial', 'test_N3_W4', 
        'test_N4_W5', 'test_N7_W5', 'test_N10_W6'
    ], 
    [
        [1,1], [1,1,1], [2,1,1], 
        [2,1,1,1], [1,1,1,0,1,0,1], [1,1,0,0,1,0,1,1,0,1]
    ]
):
    filepath = f'/nfs/users/nfs_j/jc59/quantumwork/pangenome/data/{filename}.gfa'
    graph, n, V, T = gfa_file_to_graph(filepath, copy_numbers)

    full_hamiltonian = graph_to_hubo_hamiltonian(graph, n, T, lamda=10, constraint_terms=1.0)
    hubo_hams[name] = full_hamiltonian
    
    ess = ExtendedSwapStrategy.from_all_to_all(n * T)
    num_physical_qubits = ess._num_vertices
    
    pm_rz = PassManager(
        [
            FindCommutingPauliEvolutionsMulti(), 
            CommutingGateRouter(
                ess,
                max_layers=0,
                perform_extra_swaps=True
            ),
            SwapToFinalMapping(),
            InverseCancellation(gates_to_cancel=[CXGate()]),
            CommutativeCancellation(basis_gates=["cx", "swap", "rz", "rzz"]),
            InverseCancellation(gates_to_cancel=[CXGate()]),
        ]
    )
    qc = QuantumCircuit(num_physical_qubits)
    qc.append(PauliEvolutionGate(full_hamiltonian), range(num_physical_qubits))   
    tqc_rz = pm_rz.run(qc)
    print_circuit_info(tqc_rz, 'Rz circuit')
    hubo_circs[name] = tqc_rz
    
    reordered_hamiltonian = SparsePauliOp.from_sparse_list([], tqc_rz.num_qubits)
    for instruction in tqc_rz.data:
        if instruction.operation.name == 'rz':
            qubits_str = str(instruction.qubits)
            matches = re.findall('index=([0-9]+)', qubits_str)
            if not len(matches) == 1:
                raise Exception('Bad Rz')
            reordered_hamiltonian += SparsePauliOp.from_sparse_list([('Z', [int(matches[0])], instruction.operation.params[0] / 2)], tqc_rz.num_qubits)
        if instruction.operation.name == 'PauliEvolution':
            qubits_str = str(instruction.qubits)
            matches = re.findall('index=([0-9]+)', qubits_str)
            reordered_hamiltonian += SparsePauliOp.from_sparse_list([('Z' * len(matches), [int(x) for x in matches], instruction.operation.params[0])], tqc_rz.num_qubits)
    hubo_reordered_hams[name] = reordered_hamiltonian


Keeping constraints at times: [0]
Max layers needed to apply swap decompose: 0
Gates we cannot directly implement: 0
[]
Transpiling accumulator
Rz circuit has 6 qubits
Rz circuit has 29 non-local gates and 26 non-local depth
Rz circuit contains ['PauliEvolution', 'rz'] gates.
Keeping constraints at times: [0 1]
Max layers needed to apply swap decompose: 0
Gates we cannot directly implement: 0
[]
Transpiling accumulator
Rz circuit has 9 qubits
Rz circuit has 86 non-local gates and 68 non-local depth
Rz circuit contains ['PauliEvolution', 'rz'] gates.
Keeping constraints at times: [1 0 2]
Max layers needed to apply swap decompose: 0
Gates we cannot directly implement: 0
[]
Transpiling accumulator
Rz circuit has 12 qubits
Rz circuit has 142 non-local gates and 80 non-local depth
Rz circuit contains ['PauliEvolution', 'rz'] gates.
Keeping constraints at times: [0 2 3 1]
Max layers needed to apply swap decompose: 0
Gates we cannot directly implement: 0
[]
Transpiling accumulator
Rz circuit 

In [ ]:
# from qiskit.qasm3 import dump as qasm3_dump
# for name in hubo_circs.keys():
#     with open(f'../out/ionq/qasm3/hubo_circuit_{name}.txt', 'w') as f:
#         qasm3_dump(hubo_circs[name], f)

In [14]:
# from qiskit import qasm2
# for name in hubo_circs.keys():
#     with open(f'../out/ionq/qasm2/hubo_circuit_{name}.txt', 'w') as f:
#         qasm2.dump(hubo_circs[name], f)

In [15]:
# with open('../out/ionq/hubo_hamiltonians.pkl', 'wb') as f:
#     pickle.dump(hubo_hams, f)

In [ ]:
# with open('../out/ionq/hubo_reordered_hamiltonians.pkl', 'wb') as f:
#     pickle.dump(hubo_reordered_hams, f)

In [21]:
from qiskit.qasm3 import load
hubo_circs = dict()
for name in     [
        'H_6', 'H_9', 'H_12', 
        'H_20(1)', 'H_20(2)', 'H_30'
    ]:
    hubo_circs[name] = load(f'../out/ionq/qasm3/hubo_circuit_{name}.txt')
    # with open(f'../out/ionq/qasm3/hubo_circuit_{name}.txt', 'r') as f:
        

In [24]:
hubo_circs['H_6'].draw(fold=-1)

┌───────────┐                                                                                                                                                                                                  ┌───┐┌────────────┐                         ┌───┐┌───────────┐ ┌───┐┌───────────┐┌───┐             ┌───┐┌────────────┐             ┌───┐                                                                                                                                                                                                                                                     
0 ┤ Rz(8.875) ├────────────────■─────────────────────────────────────────────────────■───────────────────────────────────■─────────────────────────────────────────────────────■─────────────────────────────────┤ X ├┤ Rz(-0.125) ├─────────────────────────┤ X ├┤ Rz(1.125) ├─┤ X ├┤ Rz(1.125) ├┤ X ├─────────────┤ X ├┤ Rz(-0.625) ├─────────────┤ X ├────────────────────────────────────■─────────────────────────────────────■───────────────────────────────────────────────■─────────────────────────────────────■────────────────────────────────────────────────────■──────────────────■────────────
  └───┬───┬───┘ ┌───────────┐┌─┴─┐┌───────────┐┌───┐┌───────────┐┌───┐┌───────────┐┌─┴─┐┌───────────┐┌───┐┌───────────┐┌─┴─┐┌───────────┐┌───┐┌───────────┐┌───┐┌───────────┐┌─┴─┐┌───────────┐┌───┐┌───────────┐└─┬─┘└───┬───┬────┘          ┌───┐┌───┐┌───┐└─┬─┘└───────────┘ └─┬─┘└───────────┘└─┬─┘┌───────────┐└─┬─┘└───┬───┬────┘┌───────────┐└─┬─┘┌───┐     ┌───┐                     │                                     │                     ┌───┐┌───────────┐ ┌───┐  │                                     │                                                    │                  │            
1 ────┤ X ├─────┤ Rz(0.625) ├┤ X ├┤ Rz(0.625) ├┤ X ├┤ Rz(0.625) ├┤ X ├┤ Rz(0.625) ├┤ X ├┤ Rz(0.625) ├┤ X ├┤ Rz(0.625) ├┤ X ├┤ Rz(0.625) ├┤ X ├┤ Rz(0.625) ├┤ X ├┤ Rz(0.625) ├┤ X ├┤ Rz(0.625) ├┤ X ├┤ Rz(0.625) ├──┼──────┤ X ├───────────────┤ X ├┤ X ├┤ X ├──■──────────────────┼─────────────────■──┤ Rz(1.125) ├──┼──────┤ X ├─────┤ Rz(1.125) ├──┼──┤ X ├─────┤ X ├─────────────────────┼─────────────────────────────────────┼─────────────────────┤ X ├┤ Rz(0.625) ├─┤ X ├──┼─────────────────────────────────────┼────────────────────────────────────────────────────┼──────────────────┼────────────
      └─┬─┘     └───────────┘└───┘└───────────┘└─┬─┘└───────────┘└─┬─┘└───────────┘├───┤└───────────┘└─┬─┘└───────────┘└───┘└───────────┘└─┬─┘└───────────┘└─┬─┘└───────────┘└───┘└───────────┘└─┬─┘└───────────┘  │      └─┬─┘     ┌───┐     └─┬─┘└─┬─┘└─┬─┘┌───┐┌────────────┐  │                    └───────────┘  │      └─┬─┘     └───────────┘  │  └─┬─┘┌───┐└─┬─┘┌───┐┌────────────┐┌─┴─┐┌────────────┐┌───┐┌────────────┐┌─┴─┐┌────────────┐┌───┐└─┬─┘└───────────┘ └─┬─┘  │                                     │                     ┌───┐     ┌───┐┌────────────┐┌─┴─┐┌────────────┐┌─┴─┐┌───┐     
2 ──────■────────────────────────────────────────┼─────────────────┼───────────────┤ X ├───────────────┼───────────────────────────────────■─────────────────┼───────────────────────────────────┼─────────────────┼────────■───────┤ X ├───────┼────■────┼──┤ X ├┤ Rz(-0.625) ├──┼───────────────────────────────────■────────┼──────────────────────■────┼──┤ X ├──┼──┤ X ├┤ Rz(-0.625) ├┤ X ├┤ Rz(-0.625) ├┤ X ├┤ Rz(-0.625) ├┤ X ├┤ Rz(-0.625) ├┤ X ├──■──────────────────■────┼─────────────────────────────────────┼─────────────────────┤ X ├─────┤ X ├┤ Rz(-0.625) ├┤ X ├┤ Rz(-0.625) ├┤ X ├┤ X ├─────
  ┌────────────┐                                 │                 │               └─┬─┘               │                                                     │                                   │                 │                └─┬─┘       │         │  └─┬─┘└────────────┘  │                                            │                           │  └─┬─┘  │  └─┬─┘└────────────┘└───┘└────────────┘└─┬─┘└────────────┘└───┘└────────────┘└─┬─┘               

In [44]:
from qiskit.qasm2 import load
hubo_circs = dict()
for name in     [
        'H_6', 'H_9', 'H_12', 
        'H_20(1)', 'H_20(2)', 'H_30'
    ]:
    hubo_circs[name] = load(f'../out/ionq/qasm2/hubo_circuit_{name}.txt')
    # with open(f'../out/ionq/qasm3/hubo_circuit_{name}.txt', 'r') as f:

In [25]:
from qiskit_aer import AerSimulator


In [29]:
ess = ExtendedSwapStrategy.from_all_to_all(3 *2)
backend = AerSimulator(coupling_map=ess._coupling_map,)

In [45]:
qc = QuantumCircuit(hubo_circs['H_6'].num_qubits)
qc.h(range(qc.num_qubits))
qc.compose(hubo_circs['H_6'], inplace=True)
qc.measure_all()
qc.draw(fold=-1)


┌───┐┌───────────┐                                                                                                                                                                                                  ┌───┐┌────────────┐                         ┌───┐┌───────────┐ ┌───┐┌───────────┐┌───┐             ┌───┐┌────────────┐             ┌───┐                                                                                                                                                                                                                                                      ░ ┌─┐               
   q_0: ┤ H ├┤ Rz(8.875) ├────────────────■─────────────────────────────────────────────────────■───────────────────────────────────■─────────────────────────────────────────────────────■─────────────────────────────────┤ X ├┤ Rz(-0.125) ├─────────────────────────┤ X ├┤ Rz(1.125) ├─┤ X ├┤ Rz(1.125) ├┤ X ├─────────────┤ X ├┤ Rz(-0.625) ├─────────────┤ X ├────────────────────────────────────■─────────────────────────────────────■───────────────────────────────────────────────■─────────────────────────────────────■────────────────────────────────────────────────────■──────────────────■─────────────░─┤M├───────────────
        ├───┤└───┬───┬───┘ ┌───────────┐┌─┴─┐┌───────────┐┌───┐┌───────────┐┌───┐┌───────────┐┌─┴─┐┌───────────┐┌───┐┌───────────┐┌─┴─┐┌───────────┐┌───┐┌───────────┐┌───┐┌───────────┐┌─┴─┐┌───────────┐┌───┐┌───────────┐└─┬─┘└───┬───┬────┘          ┌───┐┌───┐┌───┐└─┬─┘└───────────┘ └─┬─┘└───────────┘└─┬─┘┌───────────┐└─┬─┘└───┬───┬────┘┌───────────┐└─┬─┘┌───┐     ┌───┐                     │                                     │                     ┌───┐┌───────────┐ ┌───┐  │                                     │                                                    │                  │             ░ └╥┘┌─┐            
   q_1: ┤ H ├────┤ X ├─────┤ Rz(0.625) ├┤ X ├┤ Rz(0.625) ├┤ X ├┤ Rz(0.625) ├┤ X ├┤ Rz(0.625) ├┤ X ├┤ Rz(0.625) ├┤ X ├┤ Rz(0.625) ├┤ X ├┤ Rz(0.625) ├┤ X ├┤ Rz(0.625) ├┤ X ├┤ Rz(0.625) ├┤ X ├┤ Rz(0.625) ├┤ X ├┤ Rz(0.625) ├──┼──────┤ X ├───────────────┤ X ├┤ X ├┤ X ├──■──────────────────┼─────────────────■──┤ Rz(1.125) ├──┼──────┤ X ├─────┤ Rz(1.125) ├──┼──┤ X ├─────┤ X ├─────────────────────┼─────────────────────────────────────┼─────────────────────┤ X ├┤ Rz(0.625) ├─┤ X ├──┼─────────────────────────────────────┼────────────────────────────────────────────────────┼──────────────────┼─────────────░──╫─┤M├────────────
        ├───┤    └─┬─┘     └───────────┘└───┘└───────────┘└─┬─┘└───────────┘└─┬─┘└───────────┘├───┤└───────────┘└─┬─┘└───────────┘└───┘└───────────┘└─┬─┘└───────────┘└─┬─┘└───────────┘└───┘└───────────┘└─┬─┘└───────────┘  │      └─┬─┘     ┌───┐     └─┬─┘└─┬─┘└─┬─┘┌───┐┌────────────┐  │                    └───────────┘  │      └─┬─┘     └───────────┘  │  └─┬─┘┌───┐└─┬─┘┌───┐┌────────────┐┌─┴─┐┌────────────┐┌───┐┌────────────┐┌─┴─┐┌────────────┐┌───┐└─┬─┘└───────────┘ └─┬─┘  │                                     │                     ┌───┐     ┌───┐┌────────────┐┌─┴─┐┌────────────┐┌─┴─┐┌───┐      ░  ║ └╥┘┌─┐         
   q_2: ┤ H ├──────■────────────────────────────────────────┼─────────────────┼───────────────┤ X ├───────────────┼───────────────────────────────────■─────────────────┼───────────────────────────────────┼─────────────────┼────────■───────┤ X ├───────┼────■────┼──┤ X ├┤ Rz(-0.625) ├──┼───────────────────────────────────■────────┼──────────────────────■────┼──┤ X ├──┼──┤ X ├┤ Rz(-0.625) ├┤ X ├┤ Rz(-0.625) ├┤ X ├┤ Rz(-0.625) ├┤ X ├┤ Rz(-0.625) ├┤ X ├──■──────────────────■────┼─────────────────────────────────────┼─────────────────────┤ X ├─────┤ X ├┤ Rz(-0.625) ├┤ X ├┤ Rz(-0.625) ├┤ X ├┤ X ├──────░──╫──╫─┤M├─────────
        ├───┤┌────────────┐                                 │                 │               └─┬─┘               │                                                     │                                   │                 │                └─┬─┘       │         │  └─┬─┘└

In [50]:
qc.count_ops()

OrderedDict([('cx', 52), ('rz', 31), ('h', 6), ('measure', 6), ('barrier', 1)])

In [46]:
res = backend.run(qc).result()

In [47]:
res

Result(backend_name='aer_simulator', backend_version='0.17.1', job_id='49338dfc-5421-47d6-a6a2-93f3c0df1b31', success=True, results=[ExperimentResult(shots=1024, success=True, meas_level=2, data=ExperimentResultData(counts={'0x13': 6, '0x34': 8, '0x30': 8, '0x3f': 24, '0x16': 18, '0xf': 16, '0x2a': 10, '0x2c': 16, '0x24': 14, '0x14': 18, '0x17': 14, '0x3': 15, '0x25': 15, '0x3a': 18, '0xd': 16, '0x20': 28, '0x26': 17, '0x10': 12, '0x3c': 17, '0x12': 14, '0x1d': 17, '0x1a': 19, '0x1': 16, '0x1c': 11, '0x32': 25, '0xa': 15, '0x39': 22, '0x1f': 15, '0x1b': 15, '0x23': 17, '0x36': 16, '0x35': 20, '0x2': 15, '0x2b': 9, '0x8': 12, '0x3d': 21, '0xc': 9, '0x15': 19, '0x21': 18, '0x2d': 15, '0x31': 12, '0x29': 18, '0x2f': 18, '0x9': 21, '0x2e': 11, '0x0': 17, '0x1e': 12, '0x38': 11, '0x22': 20, '0xe': 13, '0x3b': 19, '0xb': 20, '0x18': 10, '0x27': 26, '0x4': 18, '0x5': 15, '0x33': 18, '0x3e': 20, '0x6': 9, '0x11': 18, '0x37': 9, '0x19': 19, '0x28': 20, '0x7': 20}), header={'creg_sizes': [['meas

In [48]:
tqc = transpile(qc, optimization_level=3)

In [49]:
tqc.draw(fold=-1)

global phase: 2.4082
        ┌───────────────┐                                                                                                                                                                                                      ┌───┐┌────────────┐                         ┌───┐┌───────────┐ ┌───┐┌───────────┐┌───┐             ┌───┐┌────────────┐             ┌───┐                                                                                                                                                                                                                                                      ░ ┌─┐               
   q_0: ┤ U2(2.5918,-π) ├────────────────────■─────────────────────────────────────────────────────■───────────────────────────────────■─────────────────────────────────────────────────────■─────────────────────────────────┤ X ├┤ Rz(-0.125) ├─────────────────────────┤ X ├┤ Rz(1.125) ├─┤ X ├┤ Rz(1.125) ├┤ X ├─────────────┤ X ├┤ Rz(-0.625) ├─────────────┤ X ├────────────────────────────────────■─────────────────────────────────────■───────────────────────────────────────────────■─────────────────────────────────────■────────────────────────────────────────────────────■──────────────────■─────────────░─┤M├───────────────
        └─────┬───┬─────┘┌───┐┌───────────┐┌─┴─┐┌───────────┐┌───┐┌───────────┐┌───┐┌───────────┐┌─┴─┐┌───────────┐┌───┐┌───────────┐┌─┴─┐┌───────────┐┌───┐┌───────────┐┌───┐┌───────────┐┌─┴─┐┌───────────┐┌───┐┌───────────┐└─┬─┘└───┬───┬────┘          ┌───┐┌───┐┌───┐└─┬─┘└───────────┘ └─┬─┘└───────────┘└─┬─┘┌───────────┐└─┬─┘└───┬───┬────┘┌───────────┐└─┬─┘┌───┐     ┌───┐                     │                                     │                     ┌───┐┌───────────┐ ┌───┐  │                                     │                                                    │                  │             ░ └╥┘┌─┐            
   q_1: ──────┤ H ├──────┤ X ├┤ Rz(0.625) ├┤ X ├┤ Rz(0.625) ├┤ X ├┤ Rz(0.625) ├┤ X ├┤ Rz(0.625) ├┤ X ├┤ Rz(0.625) ├┤ X ├┤ Rz(0.625) ├┤ X ├┤ Rz(0.625) ├┤ X ├┤ Rz(0.625) ├┤ X ├┤ Rz(0.625) ├┤ X ├┤ Rz(0.625) ├┤ X ├┤ Rz(0.625) ├──┼──────┤ X ├───────────────┤ X ├┤ X ├┤ X ├──■──────────────────┼─────────────────■──┤ Rz(1.125) ├──┼──────┤ X ├─────┤ Rz(1.125) ├──┼──┤ X ├─────┤ X ├─────────────────────┼─────────────────────────────────────┼─────────────────────┤ X ├┤ Rz(0.625) ├─┤ X ├──┼─────────────────────────────────────┼────────────────────────────────────────────────────┼──────────────────┼─────────────░──╫─┤M├────────────
              ├───┤      └─┬─┘└───────────┘└───┘└───────────┘└─┬─┘└───────────┘└─┬─┘└───────────┘├───┤└───────────┘└─┬─┘└───────────┘└───┘└───────────┘└─┬─┘└───────────┘└─┬─┘└───────────┘└───┘└───────────┘└─┬─┘└───────────┘  │      └─┬─┘     ┌───┐     └─┬─┘└─┬─┘└─┬─┘┌───┐┌────────────┐  │                    └───────────┘  │      └─┬─┘     └───────────┘  │  └─┬─┘┌───┐└─┬─┘┌───┐┌────────────┐┌─┴─┐┌────────────┐┌───┐┌────────────┐┌─┴─┐┌────────────┐┌───┐└─┬─┘└───────────┘ └─┬─┘  │                                     │                     ┌───┐     ┌───┐┌────────────┐┌─┴─┐┌────────────┐┌─┴─┐┌───┐      ░  ║ └╥┘┌─┐         
   q_2: ──────┤ H ├────────■───────────────────────────────────┼─────────────────┼───────────────┤ X ├───────────────┼───────────────────────────────────■─────────────────┼───────────────────────────────────┼─────────────────┼────────■───────┤ X ├───────┼────■────┼──┤ X ├┤ Rz(-0.625) ├──┼───────────────────────────────────■────────┼──────────────────────■────┼──┤ X ├──┼──┤ X ├┤ Rz(-0.625) ├┤ X ├┤ Rz(-0.625) ├┤ X ├┤ Rz(-0.625) ├┤ X ├┤ Rz(-0.625) ├┤ X ├──■──────────────────■────┼─────────────────────────────────────┼─────────────────────┤ X ├─────┤ X ├┤ Rz(-0.625) ├┤ X ├┤ Rz(-0.625) ├┤ X ├┤ X ├──────░──╫──╫─┤M├─────────
        ┌─────┴───┴─────┐                                      │                 │               └─┬─┘               │                                                     │                                   │               

In [51]:
with open('../out/ionq/hubo_reordered_hamiltonians.pkl', 'rb') as f:
    reorder_hams_check= pickle.load(f)

In [53]:
from qiskit.circuit.library import QAOAAnsatz

In [54]:
qaoa = QAOAAnsatz(reorder_hams_check['H_6'])

In [59]:
t_qaoa = transpile(qaoa.decompose())

In [60]:
t_qaoa.count_ops()

OrderedDict([('cx', 82), ('rz', 24), ('rzz', 7), ('h', 6), ('rx', 6)])